# 面试问题：RAPTOR 式分层摘要检索怎样实现，并避免摘要脱离原文？

**一句话回答。** 从 leaf chunk 自底向上聚类、摘要并建立树；查询先命中合适抽象层，再展开到叶子证据。每个 summary 必须带 children、源版本、摘要模型和失效状态，答案不能只引用摘要节点。

本题只用 Python 标准库重建数据合同、评分、状态机和失败分支；断言只验证小型受控样例，不能替代真实模型质量、长上下文能力或线上容量压测。

**资料入口。** [RAPTOR](https://arxiv.org/abs/2401.18059) 提出递归聚类、摘要和树状检索以整合长文档不同抽象层的信息；本例用确定性摘要代替 LLM 调用。


In [ ]:
question = "RAPTOR 分层检索"  # 执行本行的状态、计算或校验逻辑。
assert "RAPTOR" in question  # 执行本行的状态、计算或校验逻辑。
assert 4 // 2 == 2  # 执行本行的状态、计算或校验逻辑。
assert True  # 执行本行的状态、计算或校验逻辑。

## 1. leaf 是唯一原始事实层

树的叶子保存可引用的原文，内部 summary 只是导航与高层候选。每个 leaf 应有文档、范围、ACL 和版本；跨文档聚类前先确认租户和权限边界，不能让 summary 侧漏受限主题。


In [ ]:
leaves = [{"id": "l1", "raw": "已发货退款需要人工确认", "topic": "refund", "version": 1}, {"id": "l2", "raw": "退款到账通常需要三个工作日", "topic": "refund", "version": 1}, {"id": "l3", "raw": "普通订单通常三天送达", "topic": "delivery", "version": 1}]  # 执行本行的状态、计算或校验逻辑。
assert len(leaves) == 3  # 执行本行的状态、计算或校验逻辑。
assert {item["topic"] for item in leaves} == {"refund", "delivery"}  # 执行本行的状态、计算或校验逻辑。
assert all(item["raw"] for item in leaves)  # 执行本行的状态、计算或校验逻辑。

## 2. 聚类先保留成员关系，再生成摘要

真实 RAPTOR 使用 embedding cluster；教学按 topic 聚类，重点是 summary 绝不能丢 children。若聚类阈值、embedding 模型或 chunk 变化，树的 build version 必须更新。


In [ ]:
def clusters(items):  # 执行本行的状态、计算或校验逻辑。
    grouped = {}  # 执行本行的状态、计算或校验逻辑。
    for item in items:  # 执行本行的状态、计算或校验逻辑。
        grouped.setdefault(item["topic"], []).append(item["id"])  # 执行本行的状态、计算或校验逻辑。
    return grouped  # 执行本行的状态、计算或校验逻辑。
groups = clusters(leaves)  # 执行本行的状态、计算或校验逻辑。
assert groups["refund"] == ["l1", "l2"]  # 执行本行的状态、计算或校验逻辑。
assert groups["delivery"] == ["l3"]  # 执行本行的状态、计算或校验逻辑。
assert len(groups) == 2  # 执行本行的状态、计算或校验逻辑。

## 3. 摘要节点带可追溯 children 与构建版本

这里以确定性主题文本模拟摘要。生产可调用 LLM，但必须保存 prompt/model、输入 leaf ids、token budget、失败状态和 source versions；否则一段漂亮摘要无法复放或纠错。


In [ ]:
def summary_node(topic, child_ids, build_version):  # 执行本行的状态、计算或校验逻辑。
    text = "退款流程涵盖条件和到账时间" if topic == "refund" else "配送流程涵盖普通订单时效"  # 执行本行的状态、计算或校验逻辑。
    return {"id": "s_" + topic, "text": text, "children": tuple(child_ids), "build": build_version, "kind": "summary"}  # 执行本行的状态、计算或校验逻辑。
summaries = [summary_node(topic, child_ids, "tree-v1") for topic, child_ids in groups.items()]  # 执行本行的状态、计算或校验逻辑。
assert summaries[0]["children"] == ("l1", "l2")  # 执行本行的状态、计算或校验逻辑。
assert all(item["kind"] == "summary" for item in summaries)  # 执行本行的状态、计算或校验逻辑。
assert all(item["build"] == "tree-v1" for item in summaries)  # 执行本行的状态、计算或校验逻辑。

## 4. 查询可在 leaf 与 summary 两层竞争

复杂问题可能更适合命中 summary，精确条件可能更适合 leaf。正确系统会将 level、检索分数和预算显式化；不能因为 summary 排名高就直接把摘要说成已验证答案。


In [ ]:
def lexical_score(query, text):  # 执行本行的状态、计算或校验逻辑。
    return len(set(query) & set(text))  # 执行本行的状态、计算或校验逻辑。
query = "退款流程和条件"  # 执行本行的状态、计算或校验逻辑。
ranked_summaries = sorted(summaries, key=lambda item: lexical_score(query, item["text"]), reverse=True)  # 执行本行的状态、计算或校验逻辑。
assert ranked_summaries[0]["id"] == "s_refund"  # 执行本行的状态、计算或校验逻辑。
assert lexical_score(query, ranked_summaries[0]["text"]) > 0  # 执行本行的状态、计算或校验逻辑。
assert ranked_summaries[0]["kind"] == "summary"  # 执行本行的状态、计算或校验逻辑。

## 5. 命中摘要后必须展开到叶子证据

summary 的 children 是最小 provenance 路径。下游可用 query 对 child rerank，或在固定预算内挑选多个 leaf；返回的 evidence 必须仍能指出原文和版本。


In [ ]:
leaf_by_id = {item["id"]: item for item in leaves}  # 执行本行的状态、计算或校验逻辑。
def expand(summary, limit):  # 执行本行的状态、计算或校验逻辑。
    return [leaf_by_id[item_id] for item_id in summary["children"][:limit]]  # 执行本行的状态、计算或校验逻辑。
expanded = expand(ranked_summaries[0], 2)  # 执行本行的状态、计算或校验逻辑。
assert [item["id"] for item in expanded] == ["l1", "l2"]  # 执行本行的状态、计算或校验逻辑。
assert all(item["topic"] == "refund" for item in expanded)  # 执行本行的状态、计算或校验逻辑。
assert len(expand(ranked_summaries[0], 1)) == 1  # 执行本行的状态、计算或校验逻辑。

## 6. 引用保留 summary 路径，但主证据是 leaf

返回 summary id 有助于解释为何选中该分支；最终回答需要至少一个 raw leaf。多跳问题可引用多层路径，但不能只把 LLM 生成的 abstract 当唯一事实来源。


In [ ]:
def citation(summary, leaf):  # 执行本行的状态、计算或校验逻辑。
    return {"summary": summary["id"], "leaf": leaf["id"], "raw": leaf["raw"], "leaf_version": leaf["version"]}  # 执行本行的状态、计算或校验逻辑。
cite = citation(ranked_summaries[0], expanded[0])  # 执行本行的状态、计算或校验逻辑。
assert cite["summary"] == "s_refund"  # 执行本行的状态、计算或校验逻辑。
assert cite["leaf"] == "l1"  # 执行本行的状态、计算或校验逻辑。
assert "人工确认" in cite["raw"]  # 执行本行的状态、计算或校验逻辑。

## 7. leaf 更新会级联使摘要失效

更新任何 child 后，旧 summary 对应的 source-version vector 不再完整。生产应原子发布新 tree revision 或拒绝旧树，不应让新 leaf 与旧摘要混合，避免用户看到自相矛盾的上下文。


In [ ]:
def valid_summary(summary, leaves_value, expected_build):  # 执行本行的状态、计算或校验逻辑。
    ids = {item["id"] for item in leaves_value}  # 执行本行的状态、计算或校验逻辑。
    return summary["build"] == expected_build and set(summary["children"]).issubset(ids)  # 执行本行的状态、计算或校验逻辑。
assert valid_summary(ranked_summaries[0], leaves, "tree-v1")  # 执行本行的状态、计算或校验逻辑。
assert not valid_summary(ranked_summaries[0], leaves[1:], "tree-v1")  # 执行本行的状态、计算或校验逻辑。
assert not valid_summary(ranked_summaries[0], leaves, "tree-v2")  # 执行本行的状态、计算或校验逻辑。

## 8. 分层评测要检查摘要收益与错误传播

评测集应区分全局问题和精确事实问题，分别测 leaf recall、summary route accuracy、展开后证据覆盖、答案正确与树构建成本。摘要错误可能把整簇带偏，不能只报告最终某个 benchmark 的平均分。


In [ ]:
def covered(gold_ids, retrieved):  # 执行本行的状态、计算或校验逻辑。
    return set(gold_ids).issubset({item["id"] for item in retrieved})  # 执行本行的状态、计算或校验逻辑。
assert covered(["l1", "l2"], expanded)  # 执行本行的状态、计算或校验逻辑。
assert not covered(["l1", "l3"], expanded)  # 执行本行的状态、计算或校验逻辑。
assert len(cite["raw"]) > 0  # 执行本行的状态、计算或校验逻辑。

## 面试收束

面试时先说明 RAPTOR 解决“只取连续短 chunk 难理解全局”的问题，再讲 tree build、summary provenance、level-aware retrieval、展开 leaf、失效重建与分层评测。摘要是检索索引的一层，不是可脱离原文的事实权威。
